<a href="https://colab.research.google.com/github/NamithaDaney/AI-powered-paraphrasing-tool/blob/main/Assignment6_GenAi.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [22]:
pip install torch transformers sentence-transformers nltk evaluate rouge_score absl-py


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 6.0 MB/s eta 0:00:00


In [32]:
import os
import csv
import sys
import torch
import nltk
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from sentence_transformers import SentenceTransformer, util
import evaluate

# Setup NLTK resources silently
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)

class AIParaphrasingTool:
    def __init__(self):
        print("\n[1/3] Initializing Advanced AI Paraphrasing Engine...")
        self.device = "cuda" if torch.cuda.is_available() else "cpu"

        # 1. Advanced Chat-style Paraphrase Model (Significantly higher originality)
        self.para_model_name = "humarin/chatgpt_paraphraser_on_T5_base"
        self.para_tokenizer = AutoTokenizer.from_pretrained(self.para_model_name)
        self.para_model = AutoModelForSeq2SeqLM.from_pretrained(self.para_model_name).to(self.device)

        print("[2/3] Loading Semantic Evaluation Suite (SentenceTransformers)...")
        # 2. Evaluation Suite Setup
        self.similarity_model = SentenceTransformer('all-MiniLM-L6-v2', device=self.device)

        print("[3/3] Loading Lexical Overlap Suites (BLEU & ROUGE)...")
        self.bleu = evaluate.load("bleu")
        self.rouge = evaluate.load("rouge")
        print("Initialization complete! Engine is ready.\n")

    def paraphrase_text(self, text: str, temperature: float = 0.7, top_p: float = 0.95) -> str:
        """Splits text and applies clear-cut rewriting rules for high originality and clarity."""
        sentences = nltk.sent_tokenize(text)
        paraphrased_sentences = []

        for sent in sentences:
            if not sent.strip():
                continue

            # The model expects an explicit prefix formatting rule
            prompt = f"paraphrase: {sent} </s>"

            inputs = self.para_tokenizer(
                prompt,
                padding="longest",
                max_length=256,
                truncation=True,
                return_tensors="pt"
            ).to(self.device)

            # Using Diverse Beam Search / Sampling parameters tailored for humarin's model
            outputs = self.para_model.generate(
                **inputs,
                do_sample=True,
                temperature=temperature,
                top_p=top_p,
                num_return_sequences=1,
                max_length=256,
                no_repeat_ngram_size=3
            )

            rewritten_sentence = self.para_tokenizer.decode(outputs[0], skip_special_tokens=True)
            paraphrased_sentences.append(rewritten_sentence)

        return " ".join(paraphrased_sentences)

    def evaluate_output(self, original: str, paraphrased: str) -> dict:
        """Calculates exact quality, meaning preservation, and lexical shift percentages."""
        if not original.strip() or not paraphrased.strip():
            return {"semantic_similarity": 0.0, "estimated_originality": 0.0, "bleu_score": 0.0}

        # Context Meaning Check (Semantic Matching)
        emb1 = self.similarity_model.encode(original, convert_to_tensor=True)
        emb2 = self.similarity_model.encode(paraphrased, convert_to_tensor=True)
        similarity_score = float(util.pytorch_cos_sim(emb1, emb2).item())

        # Originality Check (Calculated via structural shift distance)
        bleu_results = self.bleu.compute(predictions=[paraphrased], references=[[original]])
        rouge_results = self.rouge.compute(predictions=[paraphrased], references=[[original]])
        originality_score = 1.0 - rouge_results["rougeL"]

        return {
            "semantic_similarity": round(similarity_score, 4),
            "estimated_originality": round(originality_score, 4),
            "bleu_score": round(bleu_results["bleu"], 4)
        }

    def process_batch_file(self, input_path: str, output_path: str, temp: float, top_p: float):
        """Processes a local text file sentence-by-sentence and exports a CSV report."""
        if not os.path.exists(input_path):
            print(f" Error: Input file '{input_path}' not found.")
            return

        csv_headers = ["Row ID", "Original Text", "Paraphrased Text", "Semantic Similarity", "Estimated Originality", "BLEU Score"]

        with open(input_path, "r", encoding="utf-8") as infile, \
             open(output_path, "w", encoding="utf-8", newline="") as csvfile:

            writer = csv.writer(csvfile)
            writer.writerow(csv_headers)

            print(f"\nProcessing batch from '{input_path}'...")
            for idx, line in enumerate(infile):
                clean_line = line.strip()
                if not clean_line:
                    continue

                print(f"Processing Line {idx + 1}...")
                rewritten = self.paraphrase_text(clean_line, temperature=temp, top_p=top_p)
                report = self.evaluate_output(clean_line, rewritten)

                writer.writerow([
                    idx + 1, clean_line, rewritten,
                    f"{report['semantic_similarity']:.4f}",
                    f"{report['estimated_originality']:.4f}",
                    f"{report['bleu_score']:.4f}"
                ])
        print(f" Success! Report exported completely to: '{output_path}'")


# --- Interactive Terminal User Interface ---
def main():
    print("="*60)
    print("      WELCOME TO THE AI PARAPHRASING & EVALUATION TOOL")
    print("="*60)

    # Initialize Engine
    tool = AIParaphrasingTool()

    # Base configuration hyper-parameters matching your sweet spot test run
    CREATIVITY_TEMP = 0.85
    TOP_P_NUCLEUS = 0.95

    while True:
        print("\n--- Main Console Menu ---")
        print("1. Paraphrase text interactively (Type directly into console)")
        print("2. Batch process a '.txt' file (Generates CSV Performance Report)")
        print("3. Exit Tool")

        choice = input("\nSelect an option (1-3): ").strip()

        if choice == "1":
            print("\nEnter the text you want to paraphrase (Press Enter twice on an empty line to finish):")
            lines = []
            while True:
                line = input()
                if line == "":
                    break
                lines.append(line)

            user_input = " ".join(lines).strip()
            if not user_input:
                print(" Input text cannot be empty.")
                continue

            print("\n Generating Paraphrase...")
            rewritten = tool.paraphrase_text(user_input, temperature=CREATIVITY_TEMP, top_p=TOP_P_NUCLEUS)

            print("\n" + "="*50)
            print("PARAPHRASED OUTPUT:")
            print("="*50)
            print(rewritten)

            # Print real-time analysis matrix straight to the terminal console screen
            report = tool.evaluate_output(user_input, rewritten)
            print("\n" + "="*50)
            print(" ACCURACY & QUALITY METRICS REPORT:")
            print("="*50)
            print(f" Meaning Preservation (Semantic Similarity) : {report['semantic_similarity']:.2%}")
            print(f" Lexical Originality (Vocabulary Change)   : {report['estimated_originality']:.2%}")
            print(f" Phrase Structure Overlap (BLEU Score)       : {report['bleu_score']:.4f}")
            print("="*50)

        elif choice == "2":
            input_file = input("Enter path to input text file (e.g., source_data.txt): ").strip()
            output_file = input("Enter custom name for output CSV report (default: report.csv): ").strip()
            if not output_file:
                output_file = "report.csv"

            tool.process_batch_file(input_file, output_file, CREATIVITY_TEMP, TOP_P_NUCLEUS)

        elif choice == "3":
            print("\nThank you for using the AI Paraphrasing Tool.")
            break
        else:
            print("Invalid option. Please enter a number between 1 and 3.")

if __name__ == "__main__":
    # Setup dummy text file if it does not exist to make Option 2 instantly testable
    if not os.path.exists("source_data.txt"):
        sample_paragraphs = (
            "Artificial intelligence is rapidly transforming global industry standards.\n"
            "Data scientists must adjust complex algorithms to optimize accuracy benchmarks.\n"
            "Failing to monitor systematic bias introduces massive compliance hazards issues."
        )
        with open("source_data.txt", "w", encoding="utf-8") as f:
            f.write(sample_paragraphs)

    main()


      WELCOME TO THE AI PARAPHRASING & EVALUATION TOOL

[1/3] Initializing Advanced AI Paraphrasing Engine...


Loading weights:   0%|          | 0/257 [00:00<?, ?it/s]

[2/3] Loading Semantic Evaluation Suite (SentenceTransformers)...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

[3/3] Loading Lexical Overlap Suites (BLEU & ROUGE)...
Initialization complete! Engine is ready.


--- Main Console Menu ---
1. Paraphrase text interactively (Type directly into console)
2. Batch process a '.txt' file (Generates CSV Performance Report)
3. Exit Tool

Select an option (1-3): 2
Enter path to input text file (e.g., source_data.txt): source_data.txt
Enter custom name for output CSV report (default: report.csv): 

Processing batch from 'source_data.txt'...
Processing Line 1...
Processing Line 2...
Processing Line 3...
 Success! Report exported completely to: 'report.csv'

--- Main Console Menu ---
1. Paraphrase text interactively (Type directly into console)
2. Batch process a '.txt' file (Generates CSV Performance Report)
3. Exit Tool

Select an option (1-3): 3

Thank you for using the AI Paraphrasing Tool.
